# 🐍 Master Guide: 01: Lists & Tuples Architecture

> **A Complete Foundations-to-Advanced Reference Guide**  
> Covers: Collection Categories, CPython List/Tuple Memory Internals, Complete List Method Suite, The Single-Element Tuple Comma Rule, Nested Traversal, The 3 Object Copy Paradigms, and Gotchas.
---

## 1. Core Definitions & Collection Concepts

### What is a Collection / Data Structure?
> **Definition**: A **Collection** (or Data Structure) is an object capable of storing multiple values—potentially of different, heterogeneous data types—under a single variable reference.
> In Python, collections are broadly divided into **List Category** (`list`, `tuple`), **Set Category** (`set`, `frozenset`), and **Mapping Category** (`dict`).

### What is a List?
> A **List** is a **mutable**, **ordered**, **dynamic sequence** of heterogeneous items enclosed in square brackets `[ ]`.
> - **Mutable**: Elements can be added, modified, replaced, and deleted in-place at the same memory address.
> - **Ordered**: Elements maintain insertion order, accessible via 0-based indexing and negative indexing.
> - **Dynamic**: Automatically grows and shrinks in memory as elements are added or removed.

### What is a Tuple?
> A **Tuple** is an **immutable**, **ordered sequence** of heterogeneous items enclosed in round parentheses `( )`.
> - **Immutable**: Once instantiated, its elements cannot be modified, added, or deleted.
> - **Lightweight & Safe**: Requires less memory overhead than lists; guaranteed read-only data integrity.

## 2. CPython Memory Internals: `PyListObject` vs. `PyTupleObject`

### List Dynamic Array Over-Allocation Strategy:
In CPython, a list is implemented as a variable-length array of pointers (`PyObject** ob_item`).
When appending, CPython does not allocate memory one item at a time (which would be an $O(n)$ realloc disaster). Instead, it **over-allocates** buffer capacity in pre-computed chunks:

$$\text{New Capacity} = \text{size} + (\text{size} \gg 3) + (3 \text{ if size } < 9 \text{ else } 6)$$

Growth sequence: `0, 4, 8, 16, 25, 35, 46, 58, 72, 88...` elements.
This guarantees **amortized $O(1)$ constant time** for `.append()`.

### Tuple Fixed-Allocation:
A `PyTupleObject` contains a fixed array of pointers without any dynamic resize buffers.
CPython caches empty tuples as singletons and maintains free lists for small tuples up to 20 elements, making tuple allocation faster than list allocation.

In [ ]:
# Memory Growth & Allocation Profiling (List vs. Tuple)
import sys

# 1. Inspecting dynamic list over-allocation buffer
dynamic_list = []
print("List Size Progression (Over-allocation in action):")
print("-" * 50)
prev_size = sys.getsizeof(dynamic_list)
for i in range(25):
    dynamic_list.append(i)
    current_size = sys.getsizeof(dynamic_list)
    if current_size != prev_size or i == 0:
        print(f"  Length: {len(dynamic_list):2d} items | Total Buffer Size: {current_size:3d} bytes | Allocated jump: +{current_size - prev_size} bytes")
        prev_size = current_size
print("-" * 50)

# 2. Comparing memory size of identical list vs tuple
sample_items = [10, "Python", 3.14, True, None]
lst_obj = list(sample_items)
tpl_obj = tuple(sample_items)
print(f"Memory for 5-element List  : {sys.getsizeof(lst_obj)} bytes")
print(f"Memory for 5-element Tuple : {sys.getsizeof(tpl_obj)} bytes (Tuple is leaner!)")

## 3. Complete List Operations & Predefined Methods Suite

Python lists provide an extensive built-in method suite for element insertion, removal, searching, and sorting:

| Method | Operation Performed | Time Complexity | Notes / Caveats |
|---|---|---|---|
| `.append(val)` | Appends `val` to end of list | Amortized $O(1)$ | Mutates in-place; returns `None`. |
| `.insert(idx, val)` | Inserts `val` at index `idx` | $O(n)$ | Shifts all elements after `idx` to the right. |
| `.extend(iterable)` | Unpacks and appends all items from iterable | $O(k)$ | Contrasts with `append()` which adds collection as a single nested element. |
| `.pop([idx])` | Removes and returns item at `idx` (default last) | $O(1)$ for last, $O(n)$ for arbitrary | Raises `IndexError` on empty list. |
| `.remove(val)` | Removes first occurrence of `val` | $O(n)$ | Raises `ValueError` if `val` is not found. |
| `.clear()` | Removes all elements from list | $O(n)$ | Leaves empty list `[]` in-place. |
| `.index(val, [start, stop])`| Returns index of first occurrence of `val` | $O(n)$ | Raises `ValueError` if missing. |
| `.count(val)` | Returns number of occurrences of `val` | $O(n)$ | Returns `0` if not present. |
| `.sort(key=None, reverse=False)` | Sorts list in-place using **Timsort** | $O(n \log n)$ | Stable sort. Mutates in-place; returns `None`! |
| `.reverse()` | Reverses elements in-place | $O(n)$ | Mutates in-place; returns `None`. |

In [ ]:
# Complete List Methods Execution Suite
# 1. Adding elements: append vs insert vs extend
fruits = ["Apple", "Banana"]
fruits.append("Orange")                 # Appends single element
fruits.insert(1, "Mango")               # Inserts at index 1
fruits.extend(["Grapes", "Pineapple"])   # Unpacks and appends multiple
print("After additions :", fruits)

# 2. Difference between append and extend with a collection
test_a = [1, 2]
test_b = [1, 2]
test_a.append([3, 4])  # Results in nested list: [1, 2, [3, 4]]
test_b.extend([3, 4])  # Results in flat list:   [1, 2, 3, 4]
print("append([3, 4])  :", test_a)
print("extend([3, 4])  :", test_b)

# 3. Removing elements: remove vs pop vs clear
popped_item = fruits.pop()      # Removes and returns last item
popped_first = fruits.pop(0)    # Removes and returns item at index 0
fruits.remove("Banana")         # Removes by value
print(f"Popped last: '{popped_item}', Popped first: '{popped_first}'")
print("After removals  :", fruits)

# 4. Sorting & Reversing (In-Place Timsort)
scores = [88, 42, 95, 71, 88, 63]
scores.sort(reverse=True)       # In-place descending sort
print("Scores sorted (descending):", scores)
print("Count of 88 in scores     :", scores.count(88))
print("Index of 71 in scores     :", scores.index(71))

## 4. Tuple Architecture & The Single-Element Comma Rule

### The Single-Element Tuple Trap:
In Python, parentheses `( )` are also used for arithmetic grouping.
Therefore, writing `(10)` evaluates to the **integer `10`**, NOT a tuple!
To define a single-element tuple, **a trailing comma is mandatory**: `(10,)`.

```python
x = (10)   # Type: int
y = (10,)  # Type: tuple
z = 10,    # Type: tuple (Parentheses are optional in packing!)
```

### Why Tuples Have Only 2 Methods (`count` and `index`):
Because tuples are immutable, methods that modify data (`append`, `remove`, `sort`, `clear`) do not exist.
Tuples only provide non-mutating inspection methods:
1. `tuple.count(val)`: Counts occurrences.
2. `tuple.index(val)`: Finds first index of element.

In [ ]:
# Tuple Creation, Packing, Unpacking & The Comma Rule
# 1. Verifying the Single-Element Comma Rule
not_a_tuple = ("Python")
actual_tuple = ("Python",)

print(f"('Python')   Type: {type(not_a_tuple).__name__:<6} | Value: {not_a_tuple}")
print(f"('Python',)  Type: {type(actual_tuple).__name__:<6} | Value: {actual_tuple}")

# 2. Tuple Packing and Unpacking
student_record = 101, "Suresh Kumar", "Data Science", 92.5  # Automatic packing
roll, name, course, score = student_record                   # Unpacking
print(f"\nUnpacked Profile -> ID: {roll} | Name: {name} | Course: {course} | Score: {score}")

# 3. Extended Unpacking with Asterisk (*)
first, *middle, last = (10, 20, 30, 40, 50)
print(f"Extended Unpack -> First: {first} | Middle List: {middle} | Last: {last}")

# 4. Tuple methods: count() and index()
sample_tpl = (10, 20, 30, 20, 40, 20, 50)
print(f"\nsample_tpl.count(20) : {sample_tpl.count(20)} occurrences")
print(f"sample_tpl.index(30) : Found at index {sample_tpl.index(30)}")

## 5. Nested Lists, Nested Tuples & Matrix Operations

Python supports multi-dimensional arrays via nested collections.
Elements are accessed via chained indexing: `matrix[row_idx][col_idx]`.

In [ ]:
# Matrix Operations & Nested Collection Traversal
matrix = [
    [1, 2, 3],
    [4, 5, 6],
    [7, 8, 9]
]

print("Original 3x3 Matrix:")
for row in matrix:
    print("  ", row)

print(f"\nAccessing center element matrix[1][1] : {matrix[1][1]}")
print(f"Accessing bottom-right matrix[2][2]    : {matrix[2][2]}")

# Transposing a matrix using zip(*)
transposed = [list(col) for col in zip(*matrix)]
print("\nTransposed Matrix (Columns as Rows):")
for row in transposed:
    print("  ", row)

# Flattening a 2D matrix into a 1D list
flattened = [val for row in matrix for val in row]
print("\nFlattened 1D List:", flattened)

## 6. Object Copying Techniques (The 3 Copy Paradigms)

Understanding how Python duplicates objects is essential to prevent data mutation bugs:

| Copy Technique | Syntax | Memory Behavior | Modification Impact |
|---|---|---|---|
| **1. Reference Assignment** | `b = a` | Copies **pointer only** (`id(b) == id(a)`). | Modifying `b` mutates `a` directly. |
| **2. Shallow Copy** | `b = a.copy()`, `copy.copy(a)`, `a[:]` | Creates a **new outer list**, but inner nested objects share the same memory pointers. | Modifying outer level is isolated; modifying **nested items mutates both**! |
| **3. Deep Copy** | `b = copy.deepcopy(a)` | **Recursively duplicates** outer container and all nested objects into brand new memory addresses. | Completely isolated. Modifying `b` **never** affects `a`. |

In [ ]:
# The 3 Copy Paradigms Comprehensive Drill
import copy

original = [10, 20, [30, 40]]

# Technique 1: Reference Assignment (=)
ref_copy = original

# Technique 2: Shallow Copy (.copy())
shallow_copy = original.copy()

# Technique 3: Deep Copy (copy.deepcopy())
deep_copy = copy.deepcopy(original)

# Mutating the original data
original[0] = 999       # Modifying outer element
original[2][0] = 888    # Modifying inner nested element

print("--- State After Modifying Original: original[0]=999, original[2][0]=888 ---")
print(f"Original:     {original} (Outer ID: {id(original)}, Inner ID: {id(original[2])})")
print(f"Reference:    {ref_copy} (Shares outer and inner pointers!)")
print(f"Shallow Copy: {shallow_copy} (Outer was safe, but inner nested [888, 40] was mutated!)")
print(f"Deep Copy:    {deep_copy} (COMPLETELY SAFE & ISOLATED: inner stayed [30, 40])")

## 7. Tricky Gotchas & Common Pitfalls

### ⚠️ Gotcha 1: In-Place Methods Return `None` Trap
Methods like `lst.sort()`, `lst.reverse()`, and `lst.append()` mutate the list **in-place** and return `None`.
Assigning the result back destroys your list variable!

In [ ]:
# The In-Place Return None Trap
my_list = [5, 2, 9, 1]

# TRAP (Bug!):
sorted_result = my_list.sort()
print(f"my_list.sort() returned: {sorted_result} (It returns None!)")
print(f"my_list after sort:      {my_list}")

# CORRECT:
numbers = [5, 2, 9, 1]
# Option A: In-place without assignment
numbers.sort()
# Option B: Built-in sorted() which returns a brand new list
new_sorted = sorted([5, 2, 9, 1])
print("Built-in sorted() returned new list:", new_sorted)

### ⚠️ Gotcha 2: Tuple with Mutable Nested Elements
While a tuple itself cannot be resized or have its items reassigned, if an element inside a tuple is a mutable list, that list **can still be mutated**!

In [ ]:
# Mutating a List inside a Tuple
tpl_with_list = (100, [1, 2, 3], 200)
print("Initial tuple:", tpl_with_list)

# In-place appending to the nested list
tpl_with_list[1].append(999)
print("After appending to nested list:", tpl_with_list)

# But reassigning the tuple slot itself raises TypeError:
try:
    tpl_with_list[1] = [4, 5, 6]
except TypeError as err:
    print("Direct slot reassignment FAILED:", err)

## 8. Summary & Quick Revision Cheat Sheet

| Feature | List (`list`) | Tuple (`tuple`) |
|---|---|---|
| **Syntax** | `[1, 2, 3]` | `(1, 2, 3)` (Single item: `(1,)`) |
| **Mutability** | **Mutable** (In-place modification) | **Immutable** (Read-only) |
| **Memory Buffer** | Over-allocated dynamic array | Fixed-size lean array |
| **Performance** | Slower allocation, extra pointer overhead | Faster allocation, small tuple cache |
| **Hashability** | **Unhashable** (Cannot be dict key or in set) | **Hashable** (If all its elements are hashable) |
| **Key Methods** | `.append()`, `.extend()`, `.pop()`, `.sort()`, `.remove()` | `.count()`, `.index()` |
| **Primary Use** | Dynamic data collections that change | Fixed records, database tuples, dictionary keys |